# Kernel Optimization 101 — ทำไม GPU ถึงเร็ว (และทำไมบางทีก็ไม่)

**เป้าหมาย:** เขียน CUDA kernel จริง 2 แบบ (naive กับ optimized) วัดผล แล้วเข้าใจ *ว่าอะไรคือคอขวดจริง*

แก่นที่จะได้: **คอขวดของ GPU ส่วนใหญ่ไม่ใช่การคำนวณ แต่คือการขยับข้อมูล** (memory traffic).
นี่คือเรื่องเดียวกับที่เราพิสูจน์ในโปรเจกต์ LCA ว่า CPU เอาชนะ K600 ได้ เพราะ LLM ติดคอขวดที่การขยับข้อมูล ไม่ใช่ FLOPs.

> รันบน **Google Colab** (Runtime → Change runtime type → **GPU** → T4). K600 รันไม่ได้เพราะ CUDA ใหม่เกินสถาปัตย์ Kepler.


## 0) เช็กว่ามี GPU จริงไหม


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv


## 1) แนวคิด: Arithmetic Intensity (AI)

AI = จำนวนการคำนวณ (FLOP) ต่อ 1 ไบต์ที่อ่านจาก memory.

- AI ต่ำ → โปรแกรม 'memory-bound' เอาเร็วขึ้นได้ด้วยการ **อ่าน memory ให้น้อยลง** (ไม่ใช่คำนวณให้เร็วขึ้น)
- AI สูง → 'compute-bound' ค่อยไปปรับตัวคำนวณ

Matrix multiply C = A·B (NxN) ต้องคำนวณ ~2N³ FLOP แต่ถ้าเขียนแบบ naive จะอ่าน memory ซ้ำมหาศาล.
เราจะลดการอ่านซ้ำด้วย **shared memory** (เร็วกว่า global memory ~100 เท่า) — นี่คือหัวใจของ 'tiling'.


## 2) เตรียมข้อมูล + helper วัดเวลา


In [ ]:
import numpy as np
from numba import cuda, float32
import math, time

N = 1024                      # เมทริกซ์ NxN
TPB = 16                      # threads per block ต่อแกน (tile = 16x16)

np.random.seed(0)
A = np.random.rand(N, N).astype(np.float32)
B = np.random.rand(N, N).astype(np.float32)
C_ref = A @ B                 # เฉลยจาก numpy ไว้ตรวจความถูกต้อง

def bench(fn, *args, warmup=1, iters=5):
    for _ in range(warmup): fn(*args); cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters): fn(*args)
    cuda.synchronize()
    return (time.perf_counter() - t0) / iters


## 3) Kernel แบบ NAIVE

ทุก thread คำนวณ C[i,j] หนึ่งช่อง โดยวน k แล้ว **อ่าน A กับ B จาก global memory ทุกครั้ง**.
แต่ละแถวของ A ถูกอ่านซ้ำ N ครั้ง (ครั้งละ thread ในคอลัมน์) — นี่คือ memory traffic ที่เปลืองมหาศาล.


In [ ]:
@cuda.jit
def matmul_naive(A, B, C):
    i, j = cuda.grid(2)
    if i < C.shape[0] and j < C.shape[1]:
        acc = 0.0
        for k in range(A.shape[1]):
            acc += A[i, k] * B[k, j]      # อ่าน global memory 2 ครั้งต่อรอบ k
        C[i, j] = acc

dA = cuda.to_device(A); dB = cuda.to_device(B)
dC = cuda.device_array((N, N), np.float32)
grid = (math.ceil(N / TPB), math.ceil(N / TPB))
block = (TPB, TPB)

def run_naive(): matmul_naive[grid, block](dA, dB, dC)
t_naive = bench(run_naive)
assert np.allclose(dC.copy_to_host(), C_ref, atol=1e-2)
print(f'naive : {t_naive*1e3:7.2f} ms   ({2*N**3/t_naive/1e9:6.1f} GFLOP/s)')


## 4) Kernel แบบ TILED (shared memory)

แบ่งงานเป็น tile 16x16. แต่ละ block **โหลด tile ของ A และ B เข้า shared memory ครั้งเดียว**
แล้ว thread ทั้ง 256 ตัวในบล็อกใช้ร่วมกัน → ลดการอ่าน global memory ลงราว TPB เท่า.

`cuda.syncthreads()` คือการรอให้ทุก thread โหลด tile เสร็จก่อนเริ่มคำนวณ (กัน race).


In [ ]:
@cuda.jit
def matmul_tiled(A, B, C):
    sA = cuda.shared.array((TPB, TPB), float32)
    sB = cuda.shared.array((TPB, TPB), float32)
    tx = cuda.threadIdx.x; ty = cuda.threadIdx.y
    i, j = cuda.grid(2)
    acc = 0.0
    for m in range((A.shape[1] + TPB - 1) // TPB):
        # โหลด 1 tile เข้า shared memory (ครั้งเดียว ใช้ร่วมทั้งบล็อก)
        if i < A.shape[0] and (m*TPB + tx) < A.shape[1]:
            sA[ty, tx] = A[i, m*TPB + tx]
        else: sA[ty, tx] = 0.0
        if j < B.shape[1] and (m*TPB + ty) < B.shape[0]:
            sB[ty, tx] = B[m*TPB + ty, j]
        else: sB[ty, tx] = 0.0
        cuda.syncthreads()                 # รอโหลดครบ
        for k in range(TPB):
            acc += sA[ty, k] * sB[k, tx]   # อ่านจาก shared (เร็วมาก)
        cuda.syncthreads()                 # รอคำนวณครบก่อนทับ tile ถัดไป
    if i < C.shape[0] and j < C.shape[1]:
        C[i, j] = acc

dC2 = cuda.device_array((N, N), np.float32)
def run_tiled(): matmul_tiled[grid, block](dA, dB, dC2)
t_tiled = bench(run_tiled)
assert np.allclose(dC2.copy_to_host(), C_ref, atol=1e-2)
print(f'tiled : {t_tiled*1e3:7.2f} ms   ({2*N**3/t_tiled/1e9:6.1f} GFLOP/s)')
print(f'speedup: {t_naive/t_tiled:.2f}x เร็วขึ้น (จากการ *อ่าน memory น้อยลง* ล้วนๆ)')


## 5) เทียบกับ cuBLAS (ของจริงที่ vendor ปรับมาสุด)

cuBLAS คือ library ที่ NVIDIA เขียน kernel ปรับจูนสุดๆ ไว้แล้ว — ดูว่า tiled ของเราห่างแค่ไหน
(จุดประสงค์คือเห็น 'เพดาน' ไม่ใช่เอาชนะ).


In [ ]:
try:
    import cupy as cp
    ga, gb = cp.asarray(A), cp.asarray(B)
    def run_cublas(): cp.dot(ga, gb)
    def bench_cp(fn, warmup=1, iters=5):
        for _ in range(warmup): fn(); cp.cuda.Stream.null.synchronize()
        t0=time.perf_counter()
        for _ in range(iters): fn()
        cp.cuda.Stream.null.synchronize()
        return (time.perf_counter()-t0)/iters
    t_cublas = bench_cp(run_cublas)
    print(f'cuBLAS: {t_cublas*1e3:7.2f} ms   ({2*N**3/t_cublas/1e9:6.1f} GFLOP/s)')
    print(f'tiled ได้ {t_cublas/t_tiled*100:.0f}% ของ cuBLAS (ที่เหลือคือ register tiling, double-buffer, tensor core ฯลฯ)')
except ImportError:
    print('ไม่มี cupy — ข้าม (ปกติ Colab มีให้)')


## 6) สรุป — เชื่อมกลับโปรเจกต์

1. **naive → tiled เร็วขึ้นหลายเท่า โดยไม่ได้ลดการคำนวณเลย** — ลดแค่การอ่าน memory. นี่คือหลักฐานตรงๆ ว่าคอขวดคือ data movement.
2. ลำดับขั้นการ optimize GPU: global → **shared memory (tiling)** → register tiling → double buffering → tensor cores. แต่ละขั้นคือการ 'ขยับข้อมูลให้ใกล้ที่คำนวณขึ้น'.
3. ทำไมเรื่องนี้สำคัญกับ LLM: inference ส่วนใหญ่คือ matmul ที่ **memory-bound** (อ่าน weight จาก VRAM). GPU ที่ VRAM เล็ก/ช้า (เช่น K600) เลยแพ้ แม้ FLOP จะพอ — เพราะคอขวดอยู่ที่ขยับ weight ไม่ใช่คูณเลข.
4. บทเรียนเดียวกับ `docs/HETEROGENEOUS_COMPUTE.md`: เลือกฮาร์ดแวร์/เขียนโค้ดตาม *คอขวดจริงที่วัดได้* ไม่ใช่ตามสเปคที่ดูดี.

**ลองต่อเอง:** เปลี่ยน `TPB` เป็น 8 / 32 แล้วดูผล; เปลี่ยน `N` เป็น 2048; เพิ่ม register tiling (ให้ 1 thread ทำหลายช่อง).
